# Lasso Regression (L1 Regularization)

Lasso adds L1 penalty that shrinks coefficients to EXACTLY ZERO - automatic feature selection!

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Lasso, LassoCV
from sklearn.metrics import mean_squared_error, r2_score

## 1. Load & Prepare Data

In [ ]:
X, y = load_diabetes(return_X_y=True)
feature_names = load_diabetes().feature_names
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=2)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print(f'Data: {X.shape}')

## 2. Baseline: Linear Regression

In [ ]:
lr = LinearRegression()
lr.fit(X_train_scaled, y_train)
y_pred_lr = lr.predict(X_test_scaled)
rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))
print(f'Linear - RMSE: {rmse_lr:.2f}, Features: {np.sum(lr.coef_ != 0)}/10')

## 3. Lasso Regression (alpha=1.0)

In [ ]:
lasso = Lasso(alpha=1.0)
lasso.fit(X_train_scaled, y_train)
y_pred_lasso = lasso.predict(X_test_scaled)
rmse_lasso = np.sqrt(mean_squared_error(y_test, y_pred_lasso))
n_selected = np.sum(lasso.coef_ != 0)
print(f'Lasso - RMSE: {rmse_lasso:.2f}, Features: {n_selected}/10')

## 4. Feature Selection

In [ ]:
for name, coef in zip(feature_names, lasso.coef_):
    status = 'SELECTED' if coef != 0 else 'DROPPED'
    print(f'{name}: {coef:.4f} ({status})')

## 5. Coefficient Comparison

In [ ]:
df = pd.DataFrame({'Feature': feature_names, 'Linear': lr.coef_, 'Lasso': lasso.coef_})
print(df.to_string(index=False))

## 6. Test Different Alphas

In [ ]:
results = []
for alpha in [0.01, 0.1, 0.5, 1.0, 5.0, 10.0]:
    model = Lasso(alpha=alpha, max_iter=5000)
    model.fit(X_train_scaled, y_train)
    rmse = np.sqrt(mean_squared_error(y_test, model.predict(X_test_scaled)))
    n_feat = np.sum(model.coef_ != 0)
    results.append({'alpha': alpha, 'RMSE': rmse, 'Features': n_feat})
print(pd.DataFrame(results).to_string(index=False))

## 7. Visualize

In [ ]:
results_df = pd.DataFrame(results)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(results_df['alpha'], results_df['RMSE'], 'o-', color='blue')
ax1.set_xscale('log')
ax1.set_ylabel('RMSE')
ax1.set_title('RMSE vs Alpha')
ax2.plot(results_df['alpha'], results_df['Features'], 'o-', color='green')
ax2.set_xscale('log')
ax2.set_ylabel('Features')
ax2.set_title('Features vs Alpha')
plt.tight_layout()
plt.show()

## 8. Auto-tune with Cross-Validation

In [ ]:
lasso_cv = LassoCV(cv=5, random_state=2)
lasso_cv.fit(X_train_scaled, y_train)
rmse_cv = np.sqrt(mean_squared_error(y_test, lasso_cv.predict(X_test_scaled)))
n_selected_cv = np.sum(lasso_cv.coef_ != 0)
print(f'Best alpha: {lasso_cv.alpha_:.4f}')
print(f'RMSE: {rmse_cv:.2f}, Features: {n_selected_cv}/10')